In [ ]:
# ============================================================
# CELL 1: ENVIRONMENT + GOOGLE DRIVE SETUP
# ============================================================

!pip -q install transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn networkx tqdm
!pip -q install torch-geometric

import os
import torch
import transformers
import sklearn
import pandas as pd
import numpy as np

from google.colab import drive

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

drive.mount('/content/drive')

# ------------------------------------------------------------
# Create project folders
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/FakeNewsProject"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/embeddings", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/models", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)

# ------------------------------------------------------------
# Dataset path
# ------------------------------------------------------------

DATASET_PATH = f"{PROJECT_DIR}/WELFake_Dataset.csv"

# ------------------------------------------------------------
# Environment check
# ------------------------------------------------------------

print("=" * 55)
print("ENVIRONMENT CHECK")
print("=" * 55)

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Pandas version:", pd.__version__)

print("\n" + "=" * 55)
print("GPU CHECK")
print("=" * 55)

if torch.cuda.is_available():
    print("GPU Available: YES")
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)
else:
    print("GPU Available: NO")
    print("WARNING: Enable T4 GPU in Colab.")

print("\n" + "=" * 55)
print("GOOGLE DRIVE CHECK")
print("=" * 55)

print("Project directory:", PROJECT_DIR)
print("Dataset exists:", os.path.exists(DATASET_PATH))

if os.path.exists(DATASET_PATH):
    size_mb = os.path.getsize(DATASET_PATH) / (1024 ** 2)
    print(f"Dataset size: {size_mb:.2f} MB")
else:
    print("ERROR: WELFake_Dataset.csv not found in Google Drive.")

print("\nSetup completed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.3 MB/s eta 0:00:00
Mounted at /content/drive
ENVIRONMENT CHECK
PyTorch version: 2.11.0+cu128
Transformers version: 5.13.1
Scikit-learn version: 1.6.1
Pandas version: 2.2.2

GPU CHECK
GPU Available: YES
GPU: Tesla T4
CUDA Version: 12.8

GOOGLE DRIVE CHECK
Project directory: /content/drive/MyDrive/FakeNewsProject
Dataset exists: True
Dataset size: 233.73 MB

Setup completed.


In [ ]:
# ============================================================
# CELL 2: LOAD AND INSPECT WELFAKE DATASET
# ============================================================

print("=" * 55)
print("LOADING WELFAKE DATASET")
print("=" * 55)

df = pd.read_csv(
    DATASET_PATH,
    low_memory=False
)

print("Dataset loaded successfully!")

print("\n" + "=" * 55)
print("DATASET DIMENSIONS")
print("=" * 55)

print("Number of articles:", len(df))
print("Number of columns:", len(df.columns))

print("\n" + "=" * 55)
print("COLUMN NAMES")
print("=" * 55)

print(df.columns.tolist())

print("\n" + "=" * 55)
print("FIRST 5 RECORDS")
print("=" * 55)

display(df.head())

print("\n" + "=" * 55)
print("DATA TYPES")
print("=" * 55)

print(df.dtypes)

LOADING WELFAKE DATASET
Dataset loaded successfully!

DATASET DIMENSIONS
Number of articles: 72134
Number of columns: 4

COLUMN NAMES
['Unnamed: 0', 'title', 'text', 'label']

FIRST 5 RECORDS


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1



DATA TYPES
Unnamed: 0     int64
title         object
text          object
label          int64
dtype: object


In [ ]:
# ============================================================
# CELL 3: DATASET QUALITY ANALYSIS
# ============================================================

print("=" * 55)
print("MISSING VALUES")
print("=" * 55)

missing_values = df.isnull().sum()
print(missing_values)

print("\n" + "=" * 55)
print("MISSING VALUE PERCENTAGE")
print("=" * 55)

missing_percentage = (df.isnull().sum() / len(df)) * 100
print(missing_percentage.round(2))

print("\n" + "=" * 55)
print("DUPLICATE ANALYSIS")
print("=" * 55)

print("Duplicate rows:", df.duplicated().sum())

print("\n" + "=" * 55)
print("LABEL DISTRIBUTION")
print("=" * 55)

label_counts = df['label'].value_counts().sort_index()
print(label_counts)

print("\n" + "=" * 55)
print("LABEL PERCENTAGE")
print("=" * 55)

label_percentage = df['label'].value_counts(
    normalize=True
).sort_index() * 100

print(label_percentage.round(2))

MISSING VALUES
Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

MISSING VALUE PERCENTAGE
Unnamed: 0    0.00
title         0.77
text          0.05
label         0.00
dtype: float64

DUPLICATE ANALYSIS
Duplicate rows: 0

LABEL DISTRIBUTION
label
0    35028
1    37106
Name: count, dtype: int64

LABEL PERCENTAGE
label
0    48.56
1    51.44
Name: proportion, dtype: float64


In [ ]:
# ============================================================
# CELL 4: DATA CLEANING AND MODEL INPUT CREATION
# ============================================================

print("=" * 55)
print("DATA CLEANING")
print("=" * 55)

print("Rows before cleaning:", len(df))

# ------------------------------------------------------------
# 1. Remove unnecessary index column
# ------------------------------------------------------------

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# ------------------------------------------------------------
# 2. Remove records with missing article text
# ------------------------------------------------------------

df = df.dropna(subset=['text'])

# ------------------------------------------------------------
# 3. Replace missing titles with empty string
# ------------------------------------------------------------

df['title'] = df['title'].fillna('')

# ------------------------------------------------------------
# 4. Ensure correct data types
# ------------------------------------------------------------

df['title'] = df['title'].astype(str)
df['text'] = df['text'].astype(str)
df['label'] = df['label'].astype(int)

# ------------------------------------------------------------
# 5. Combine title + article text
# ------------------------------------------------------------

df['content'] = df['title'] + " " + df['text']

# ------------------------------------------------------------
# 6. Reset index
# ------------------------------------------------------------

df = df.reset_index(drop=True)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("CLEANING COMPLETE")
print("=" * 55)

print("Rows after cleaning:", len(df))
print("Columns:", df.columns.tolist())

print("\nRemaining missing values:")
print(df[['title', 'text', 'label']].isnull().sum())

print("\nSample combined input:")
print("-" * 55)
print(df.loc[0, 'content'][:1000])

print("\nLabel:", df.loc[0, 'label'])

DATA CLEANING
Rows before cleaning: 72134

CLEANING COMPLETE
Rows after cleaning: 72095
Columns: ['title', 'text', 'label', 'content']

Remaining missing values:
title    0
text     0
label    0
dtype: int64

Sample combined input:
-------------------------------------------------------
LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO] No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for the lynching and hanging of white people and cops. They encouraged others on a radio show Tuesday night to  turn the tide  and kill white people and cops to send a message about the killing of black people in America.One of the F***YoFlag organizers is called  Sunshine.  She has a radio blog show hosted from Texas called,  Sunshine s F***ing Opinion Radio Show. A snapshot of her #FYF911 @LOLatWhiteFear Twitter page at 9:53 p.m. shows that she was urging su

In [ ]:
# ============================================================
# CELL 5: CREATE STRATIFIED EXPERIMENTAL DATASET
# ============================================================

from sklearn.model_selection import train_test_split

SAMPLES_PER_CLASS = 5000

# ------------------------------------------------------------
# Separate Fake and Real articles
# ------------------------------------------------------------

fake_df = df[df['label'] == 0]
real_df = df[df['label'] == 1]

print("Available Fake articles:", len(fake_df))
print("Available Real articles:", len(real_df))

# ------------------------------------------------------------
# Randomly select equal samples from both classes
# ------------------------------------------------------------

fake_sample = fake_df.sample(
    n=SAMPLES_PER_CLASS,
    random_state=42
)

real_sample = real_df.sample(
    n=SAMPLES_PER_CLASS,
    random_state=42
)

# ------------------------------------------------------------
# Combine and shuffle
# ------------------------------------------------------------

experiment_df = pd.concat(
    [fake_sample, real_sample],
    ignore_index=True
)

experiment_df = experiment_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# ------------------------------------------------------------
# Save experimental dataset to Google Drive
# ------------------------------------------------------------

EXPERIMENT_PATH = f"{PROJECT_DIR}/data/welFake_experiment_10000.csv"

experiment_df.to_csv(
    EXPERIMENT_PATH,
    index=False
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("EXPERIMENTAL DATASET")
print("=" * 55)

print("Total samples:", len(experiment_df))

print("\nClass distribution:")
print(experiment_df['label'].value_counts())

print("\nClass percentage:")
print(
    (experiment_df['label'].value_counts(normalize=True) * 100)
    .round(2)
)

print("\nSaved to:")
print(EXPERIMENT_PATH)

print("\nFile exists:", os.path.exists(EXPERIMENT_PATH))

Available Fake articles: 35028
Available Real articles: 37067

EXPERIMENTAL DATASET
Total samples: 10000

Class distribution:
label
1    5000
0    5000
Name: count, dtype: int64

Class percentage:
label
1    50.0
0    50.0
Name: proportion, dtype: float64

Saved to:
/content/drive/MyDrive/FakeNewsProject/data/welFake_experiment_10000.csv

File exists: True


In [ ]:
# ============================================================
# CELL 6: TRAIN / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# Stratified 80/20 split
# ------------------------------------------------------------

train_df, test_df = train_test_split(
    experiment_df,
    test_size=0.20,
    random_state=42,
    stratify=experiment_df['label']
)

# Reset indices
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# ------------------------------------------------------------
# File paths
# ------------------------------------------------------------

TRAIN_PATH = f"{PROJECT_DIR}/data/welFake_train_8000.csv"
TEST_PATH = f"{PROJECT_DIR}/data/welFake_test_2000.csv"

# ------------------------------------------------------------
# Save to Google Drive
# ------------------------------------------------------------

train_df.to_csv(TRAIN_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 55)
print("TRAIN / TEST SPLIT")
print("=" * 55)

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))

print("\nTraining distribution:")
print(train_df['label'].value_counts().sort_index())

print("\nTesting distribution:")
print(test_df['label'].value_counts().sort_index())

print("\nTraining percentages:")
print(
    (train_df['label'].value_counts(normalize=True) * 100)
    .sort_index()
    .round(2)
)

print("\nTesting percentages:")
print(
    (test_df['label'].value_counts(normalize=True) * 100)
    .sort_index()
    .round(2)
)

print("\nFiles saved:")
print(TRAIN_PATH)
print(TEST_PATH)

print("\nTrain file exists:", os.path.exists(TRAIN_PATH))
print("Test file exists:", os.path.exists(TEST_PATH))

TRAIN / TEST SPLIT
Training samples: 8000
Testing samples: 2000

Training distribution:
label
0    4000
1    4000
Name: count, dtype: int64

Testing distribution:
label
0    1000
1    1000
Name: count, dtype: int64

Training percentages:
label
0    50.0
1    50.0
Name: proportion, dtype: float64

Testing percentages:
label
0    50.0
1    50.0
Name: proportion, dtype: float64

Files saved:
/content/drive/MyDrive/FakeNewsProject/data/welFake_train_8000.csv
/content/drive/MyDrive/FakeNewsProject/data/welFake_test_2000.csv

Train file exists: True
Test file exists: True


In [ ]:
# ============================================================
# CELL 7: LOAD DISTILBERT (LLM)
# ============================================================

from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "distilbert-base-uncased"

print("=" * 55)
print("LOADING DISTILBERT")
print("=" * 55)

# ------------------------------------------------------------
# Load tokenizer
# ------------------------------------------------------------

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# ------------------------------------------------------------
# Load pretrained Transformer
# ------------------------------------------------------------

print("Loading model...")

llm_model = AutoModel.from_pretrained(
    MODEL_NAME
)

# ------------------------------------------------------------
# Move model to GPU
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

llm_model = llm_model.to(device)

# ------------------------------------------------------------
# Evaluation mode
# ------------------------------------------------------------

llm_model.eval()

# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("LLM CONFIGURATION")
print("=" * 55)

print("Model:", MODEL_NAME)
print("Device:", device)
print("Hidden size:", llm_model.config.hidden_size)
print("Maximum sequence length:", tokenizer.model_max_length)

LOADING DISTILBERT
Loading tokenizer...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



LLM CONFIGURATION
Model: distilbert-base-uncased
Device: cuda
Hidden size: 768
Maximum sequence length: 512


In [ ]:
# ============================================================
# CELL 8: TEST DISTILBERT EMBEDDING
# ============================================================

sample_text = train_df.loc[0, "content"]

# Tokenize
inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=256,
    padding=True
)

# Move to GPU
inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

# Generate contextual representation
with torch.no_grad():
    outputs = llm_model(**inputs)

# Attention-mask-aware mean pooling
attention_mask = inputs["attention_mask"]

token_embeddings = outputs.last_hidden_state

mask = attention_mask.unsqueeze(-1).expand(
    token_embeddings.size()
).float()

sum_embeddings = torch.sum(
    token_embeddings * mask,
    dim=1
)

sum_mask = torch.clamp(
    mask.sum(dim=1),
    min=1e-9
)

embedding = sum_embeddings / sum_mask

print("=" * 55)
print("LLM EMBEDDING TEST")
print("=" * 55)

print("Input characters:", len(sample_text))
print("Tokenized shape:", inputs["input_ids"].shape)
print("Embedding shape:", embedding.shape)
print("Embedding device:", embedding.device)

print("\nFirst 10 embedding values:")
print(embedding[0][:10])

LLM EMBEDDING TEST
Input characters: 2108
Tokenized shape: torch.Size([1, 256])
Embedding shape: torch.Size([1, 768])
Embedding device: cuda:0

First 10 embedding values:
tensor([-0.3336, -0.2822,  0.1719,  0.0759,  0.1690, -0.0873, -0.0665,  0.1916,
        -0.1669, -0.0514], device='cuda:0')


In [ ]:
# ============================================================
# CELL 9: GENERATE AND SAVE DISTILBERT EMBEDDINGS
# ============================================================

from tqdm.auto import tqdm
import numpy as np
import torch
import os

BATCH_SIZE = 32
MAX_LENGTH = 256

texts = experiment_df["content"].tolist()

EMBEDDING_PATH = (
    f"{PROJECT_DIR}/embeddings/"
    "distilbert_embeddings.npy"
)

print("=" * 55)
print("DISTILBERT EMBEDDING GENERATION")
print("=" * 55)

print("Total articles:", len(texts))
print("Batch size:", BATCH_SIZE)
print("Maximum tokens:", MAX_LENGTH)
print("Device:", device)

# ------------------------------------------------------------
# Attention-mask-aware mean pooling
# ------------------------------------------------------------

def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    sum_embeddings = torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    )

    sum_mask = torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

    return sum_embeddings / sum_mask


# ------------------------------------------------------------
# Generate embeddings in batches
# ------------------------------------------------------------

all_embeddings = []

for start_idx in tqdm(
    range(0, len(texts), BATCH_SIZE),
    desc="Generating DistilBERT embeddings"
):

    batch_texts = texts[
        start_idx:start_idx + BATCH_SIZE
    ]

    encoded = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    # Move tensors to GPU
    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.no_grad():

        outputs = llm_model(**encoded)

        batch_embeddings = mean_pooling(
            outputs,
            encoded["attention_mask"]
        )

    # Move result to CPU
    all_embeddings.append(
        batch_embeddings.cpu().numpy()
    )

    # Free GPU memory
    del encoded
    del outputs
    del batch_embeddings

# ------------------------------------------------------------
# Combine all batches
# ------------------------------------------------------------

embeddings = np.vstack(all_embeddings)

print("\n" + "=" * 55)
print("EMBEDDINGS GENERATED")
print("=" * 55)

print("Embedding matrix shape:", embeddings.shape)
print("Data type:", embeddings.dtype)

# ------------------------------------------------------------
# SAVE DIRECTLY TO GOOGLE DRIVE
# ------------------------------------------------------------

np.save(
    EMBEDDING_PATH,
    embeddings
)

print("\n" + "=" * 55)
print("EMBEDDINGS SAVED")
print("=" * 55)

print("Path:", EMBEDDING_PATH)
print("File exists:", os.path.exists(EMBEDDING_PATH))

if os.path.exists(EMBEDDING_PATH):
    size_mb = os.path.getsize(EMBEDDING_PATH) / (1024 ** 2)
    print(f"File size: {size_mb:.2f} MB")

DISTILBERT EMBEDDING GENERATION
Total articles: 10000
Batch size: 32
Maximum tokens: 256
Device: cuda


Generating DistilBERT embeddings:   0%|          | 0/313 [00:00<?, ?it/s]


EMBEDDINGS GENERATED
Embedding matrix shape: (10000, 768)
Data type: float32

EMBEDDINGS SAVED
Path: /content/drive/MyDrive/FakeNewsProject/embeddings/distilbert_embeddings.npy
File exists: True
File size: 29.30 MB


In [ ]:
# ============================================================
# CELL 10: LOAD AND VERIFY SAVED EMBEDDINGS
# ============================================================

import numpy as np
import os

EMBEDDING_PATH = (
    f"{PROJECT_DIR}/embeddings/"
    "distilbert_embeddings.npy"
)

# Load embeddings directly from Google Drive
embeddings = np.load(EMBEDDING_PATH)

print("=" * 55)
print("SAVED EMBEDDING VERIFICATION")
print("=" * 55)

print("Embedding shape:", embeddings.shape)
print("Data type:", embeddings.dtype)
print("Number of articles:", embeddings.shape[0])
print("Embedding dimension:", embeddings.shape[1])

print("\nFile exists:", os.path.exists(EMBEDDING_PATH))

print("\nFirst 10 values of first embedding:")
print(embeddings[0][:10])

SAVED EMBEDDING VERIFICATION
Embedding shape: (10000, 768)
Data type: float32
Number of articles: 10000
Embedding dimension: 768

File exists: True

First 10 values of first embedding:
[-0.03427679  0.0177361   0.34359628 -0.10044958  0.14234802 -0.02512578
  0.04751306  0.4228208  -0.04528466 -0.25534025]


In [ ]:
# ============================================================
# CELL 11: BUILD NEWS SIMILARITY GRAPH
# ============================================================

from sklearn.neighbors import NearestNeighbors
import numpy as np
import os

# Number of neighbors
K = 10

print("=" * 55)
print("BUILDING NEWS SIMILARITY GRAPH")
print("=" * 55)

print("Number of nodes:", embeddings.shape[0])
print("Node feature dimension:", embeddings.shape[1])
print("Number of nearest neighbors:", K)
print("Similarity metric: Cosine")

# ------------------------------------------------------------
# Build k-Nearest Neighbors model
# ------------------------------------------------------------

knn = NearestNeighbors(
    n_neighbors=K + 1,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

knn.fit(embeddings)

# Find nearest neighbors
distances, neighbor_indices = knn.kneighbors(embeddings)

# ------------------------------------------------------------
# Remove self-neighbor
# ------------------------------------------------------------

# First neighbor is always the node itself
neighbor_indices = neighbor_indices[:, 1:]
distances = distances[:, 1:]

# Convert cosine distance to cosine similarity
similarities = 1 - distances

# ------------------------------------------------------------
# Display graph statistics
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("GRAPH CONSTRUCTION COMPLETE")
print("=" * 55)

print("Neighbor matrix shape:", neighbor_indices.shape)
print("Distance matrix shape:", distances.shape)
print("Similarity matrix shape:", similarities.shape)

print("\nAverage neighbor similarity:",
      round(float(similarities.mean()), 4))

print("Minimum neighbor similarity:",
      round(float(similarities.min()), 4))

print("Maximum neighbor similarity:",
      round(float(similarities.max()), 4))

print("\nExample:")
print("Article 0 nearest neighbors:",
      neighbor_indices[0])

print("\nArticle 0 similarities:")
print(np.round(similarities[0], 4))

BUILDING NEWS SIMILARITY GRAPH
Number of nodes: 10000
Node feature dimension: 768
Number of nearest neighbors: 10
Similarity metric: Cosine

GRAPH CONSTRUCTION COMPLETE
Neighbor matrix shape: (10000, 10)
Distance matrix shape: (10000, 10)
Similarity matrix shape: (10000, 10)

Average neighbor similarity: 0.9489
Minimum neighbor similarity: 0.5772
Maximum neighbor similarity: 1.0

Example:
Article 0 nearest neighbors: [4197 6191 5121 7614 1194  935 4614  717 8405 4500]

Article 0 similarities:
[0.9253 0.9234 0.9227 0.922  0.9215 0.9205 0.9204 0.9202 0.9186 0.918 ]


In [ ]:
# ============================================================
# CELL 12: CREATE PYTORCH GEOMETRIC GRAPH
# ============================================================

import torch
from torch_geometric.data import Data

print("=" * 55)
print("CREATING PYTORCH GEOMETRIC GRAPH")
print("=" * 55)

# ------------------------------------------------------------
# Create directed edges from kNN
# ------------------------------------------------------------

source_nodes = np.repeat(
    np.arange(len(embeddings)),
    K
)

target_nodes = neighbor_indices.reshape(-1)

edge_weights = similarities.reshape(-1)

# ------------------------------------------------------------
# Convert to undirected graph
# ------------------------------------------------------------

source_undirected = np.concatenate([
    source_nodes,
    target_nodes
])

target_undirected = np.concatenate([
    target_nodes,
    source_nodes
])

edge_weights_undirected = np.concatenate([
    edge_weights,
    edge_weights
])

# ------------------------------------------------------------
# Create edge_index
# ------------------------------------------------------------

edge_index = torch.tensor(
    np.vstack([
        source_undirected,
        target_undirected
    ]),
    dtype=torch.long
)

# ------------------------------------------------------------
# Create edge weights
# ------------------------------------------------------------

edge_weight = torch.tensor(
    edge_weights_undirected,
    dtype=torch.float
)

# ------------------------------------------------------------
# Node features
# ------------------------------------------------------------

x = torch.tensor(
    embeddings,
    dtype=torch.float
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

y = torch.tensor(
    experiment_df["label"].values,
    dtype=torch.long
)

# ------------------------------------------------------------
# Create train/test masks
# ------------------------------------------------------------

train_indices = train_df.index.values
test_indices = test_df.index.values

train_mask = torch.zeros(
    len(experiment_df),
    dtype=torch.bool
)

test_mask = torch.zeros(
    len(experiment_df),
    dtype=torch.bool
)

train_mask[train_indices] = True
test_mask[test_indices] = True

# ------------------------------------------------------------
# Create PyTorch Geometric Data object
# ------------------------------------------------------------

graph_data = Data(
    x=x,
    edge_index=edge_index,
    edge_weight=edge_weight,
    y=y,
    train_mask=train_mask,
    test_mask=test_mask
)

print("\n" + "=" * 55)
print("GRAPH CREATED")
print("=" * 55)

print(graph_data)

print("\nNumber of nodes:", graph_data.num_nodes)
print("Number of edges:", graph_data.num_edges)
print("Node feature size:", graph_data.x.shape)
print("Labels:", graph_data.y.shape)

print("\nTrain nodes:", int(graph_data.train_mask.sum()))
print("Test nodes:", int(graph_data.test_mask.sum()))

print("\nEdge weight shape:", graph_data.edge_weight.shape)

CREATING PYTORCH GEOMETRIC GRAPH

GRAPH CREATED
Data(x=[10000, 768], edge_index=[2, 200000], y=[10000], edge_weight=[200000], train_mask=[10000], test_mask=[10000])

Number of nodes: 10000
Number of edges: 200000
Node feature size: torch.Size([10000, 768])
Labels: torch.Size([10000])

Train nodes: 8000
Test nodes: 2000

Edge weight shape: torch.Size([200000])


In [ ]:
# ============================================================
# CELL 13: DEFINE GRAPHSAGE MODEL
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import SAGEConv

# ------------------------------------------------------------
# GraphSAGE Model
# ------------------------------------------------------------

class GraphSAGE(nn.Module):

    def __init__(
        self,
        input_dim=768,
        hidden_dim=128,
        output_dim=2,
        dropout=0.3
    ):
        super().__init__()

        # First GraphSAGE layer
        self.conv1 = SAGEConv(
            input_dim,
            hidden_dim
        )

        # Second GraphSAGE layer
        self.conv2 = SAGEConv(
            hidden_dim,
            64
        )

        # Final classifier
        self.classifier = nn.Linear(
            64,
            output_dim
        )

        self.dropout = dropout

    def forward(self, x, edge_index):

        # Layer 1
        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # Layer 2
        x = self.conv2(
            x,
            edge_index
        )

        x = F.relu(x)

        # Classification
        x = self.classifier(x)

        return x


# ------------------------------------------------------------
# Initialize model
# ------------------------------------------------------------

gnn_model = GraphSAGE(
    input_dim=768,
    hidden_dim=128,
    output_dim=2,
    dropout=0.3
)

# Move to GPU
gnn_model = gnn_model.to(device)

print("=" * 55)
print("GRAPHSAGE MODEL")
print("=" * 55)

print(gnn_model)

print("\nModel device:", device)

# Count parameters
total_params = sum(
    p.numel()
    for p in gnn_model.parameters()
)

print("Total trainable parameters:", total_params)

GRAPHSAGE MODEL
GraphSAGE(
  (conv1): SAGEConv(768, 128, aggr=mean)
  (conv2): SAGEConv(128, 64, aggr=mean)
  (classifier): Linear(in_features=64, out_features=2, bias=True)
)

Model device: cuda
Total trainable parameters: 213314


In [ ]:
# ============================================================
# CELL 14: PREPARE GRAPHSAGE TRAINING
# ============================================================

import torch
import torch.nn.functional as F

# Move graph to GPU
graph_data = graph_data.to(device)

# Re-create model on GPU
gnn_model = GraphSAGE(
    input_dim=768,
    hidden_dim=128,
    output_dim=2,
    dropout=0.3
).to(device)

# Optimizer
optimizer = torch.optim.Adam(
    gnn_model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

# Loss function
criterion = torch.nn.CrossEntropyLoss()

print("=" * 55)
print("GRAPHSAGE TRAINING SETUP")
print("=" * 55)

print("Graph device:", graph_data.x.device)
print("Model device:", next(gnn_model.parameters()).device)

print("Training nodes:", int(graph_data.train_mask.sum()))
print("Testing nodes:", int(graph_data.test_mask.sum()))

print("Learning rate:", 0.001)
print("Weight decay:", 5e-4)
print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam")

GRAPHSAGE TRAINING SETUP
Graph device: cuda:0
Model device: cuda:0
Training nodes: 8000
Testing nodes: 2000
Learning rate: 0.001
Weight decay: 0.0005
Loss function: CrossEntropyLoss
Optimizer: Adam


In [ ]:
# ============================================================
# CELL 15: TRAIN GRAPHSAGE
# ============================================================

import time

EPOCHS = 50

train_losses = []
train_accuracies = []

print("=" * 55)
print("GRAPHSAGE TRAINING")
print("=" * 55)

start_time = time.time()

for epoch in range(1, EPOCHS + 1):

    # --------------------------------------------------------
    # Training mode
    # --------------------------------------------------------

    gnn_model.train()

    # Clear previous gradients
    optimizer.zero_grad()

    # Forward pass
    output = gnn_model(
        graph_data.x,
        graph_data.edge_index
    )

    # Calculate loss only on training nodes
    loss = criterion(
        output[graph_data.train_mask],
        graph_data.y[graph_data.train_mask]
    )

    # Backpropagation
    loss.backward()

    # Update model parameters
    optimizer.step()

    # --------------------------------------------------------
    # Training accuracy
    # --------------------------------------------------------

    predictions = output.argmax(dim=1)

    train_correct = (
        predictions[graph_data.train_mask]
        == graph_data.y[graph_data.train_mask]
    ).sum().item()

    train_total = int(graph_data.train_mask.sum())

    train_accuracy = (
        train_correct / train_total
    )

    train_losses.append(loss.item())
    train_accuracies.append(train_accuracy)

    # --------------------------------------------------------
    # Print progress
    # --------------------------------------------------------

    if epoch == 1 or epoch % 5 == 0:

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Loss: {loss.item():.4f} | "
            f"Train Accuracy: {train_accuracy:.4f}"
        )

training_time = time.time() - start_time

print("\n" + "=" * 55)
print("TRAINING COMPLETE")
print("=" * 55)

print(
    f"Training time: "
    f"{training_time:.2f} seconds"
)

print(
    f"Final training loss: "
    f"{train_losses[-1]:.4f}"
)

print(
    f"Final training accuracy: "
    f"{train_accuracies[-1] * 100:.2f}%"
)

GRAPHSAGE TRAINING
Epoch 01/50 | Loss: 0.6931 | Train Accuracy: 0.5056
Epoch 05/50 | Loss: 0.5930 | Train Accuracy: 0.7984
Epoch 10/50 | Loss: 0.4378 | Train Accuracy: 0.8101
Epoch 15/50 | Loss: 0.3972 | Train Accuracy: 0.8230
Epoch 20/50 | Loss: 0.3560 | Train Accuracy: 0.8441
Epoch 25/50 | Loss: 0.3185 | Train Accuracy: 0.8672
Epoch 30/50 | Loss: 0.3037 | Train Accuracy: 0.8762
Epoch 35/50 | Loss: 0.2853 | Train Accuracy: 0.8805
Epoch 40/50 | Loss: 0.2680 | Train Accuracy: 0.8885
Epoch 45/50 | Loss: 0.2555 | Train Accuracy: 0.8948
Epoch 50/50 | Loss: 0.2471 | Train Accuracy: 0.9011

TRAINING COMPLETE
Training time: 1.71 seconds
Final training loss: 0.2471
Final training accuracy: 90.11%


In [ ]:
# ============================================================
# CELL 16: EVALUATE GRAPHSAGE ON TEST DATA
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# Evaluation mode
# ------------------------------------------------------------

gnn_model.eval()

with torch.no_grad():

    test_output = gnn_model(
        graph_data.x,
        graph_data.edge_index
    )

    test_predictions = test_output.argmax(
        dim=1
    )

# ------------------------------------------------------------
# Extract test labels and predictions
# ------------------------------------------------------------

y_test = (
    graph_data.y[graph_data.test_mask]
    .cpu()
    .numpy()
)

y_pred = (
    test_predictions[graph_data.test_mask]
    .cpu()
    .numpy()
)

# ------------------------------------------------------------
# Calculate metrics
# ------------------------------------------------------------

test_accuracy = accuracy_score(
    y_test,
    y_pred
)

test_precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 55)
print("GRAPHSAGE TEST RESULTS")
print("=" * 55)

print(f"Accuracy : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Precision: {test_precision:.4f} ({test_precision*100:.2f}%)")
print(f"Recall   : {test_recall:.4f} ({test_recall*100:.2f}%)")
print(f"F1 Score : {test_f1:.4f} ({test_f1*100:.2f}%)")

print("\n" + "=" * 55)
print("CLASSIFICATION REPORT")
print("=" * 55)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Fake", "Real"],
        zero_division=0
    )
)

GRAPHSAGE TEST RESULTS
Accuracy : 0.9065 (90.65%)
Precision: 0.9229 (92.29%)
Recall   : 0.8846 (88.46%)
F1 Score : 0.9034 (90.34%)

CLASSIFICATION REPORT
              precision    recall  f1-score   support

        Fake       0.89      0.93      0.91      1012
        Real       0.92      0.88      0.90       988

    accuracy                           0.91      2000
   macro avg       0.91      0.91      0.91      2000
weighted avg       0.91      0.91      0.91      2000



In [ ]:
print("Experiment distribution:")
print(experiment_df["label"].value_counts().sort_index())

print("\nTrain distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nTest distribution:")
print(test_df["label"].value_counts().sort_index())

print("\nGraph train-mask distribution:")
print(
    pd.Series(
        graph_data.y[graph_data.train_mask].cpu().numpy()
    ).value_counts().sort_index()
)

print("\nGraph test-mask distribution:")
print(
    pd.Series(
        graph_data.y[graph_data.test_mask].cpu().numpy()
    ).value_counts().sort_index()
)

Experiment distribution:
label
0    5000
1    5000
Name: count, dtype: int64

Train distribution:
label
0    4000
1    4000
Name: count, dtype: int64

Test distribution:
label
0    1000
1    1000
Name: count, dtype: int64

Graph train-mask distribution:
0    4020
1    3980
Name: count, dtype: int64

Graph test-mask distribution:
0    1012
1     988
Name: count, dtype: int64


In [ ]:
# ============================================================
# FIX: CORRECT TRAIN / TEST NODE MAPPING
# ============================================================

from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# Create node indices
# ------------------------------------------------------------

node_indices = np.arange(len(experiment_df))
node_labels = experiment_df["label"].values

# ------------------------------------------------------------
# Stratified split using ORIGINAL experiment_df positions
# ------------------------------------------------------------

train_node_indices, test_node_indices = train_test_split(
    node_indices,
    test_size=0.20,
    random_state=42,
    stratify=node_labels
)

# ------------------------------------------------------------
# Create correct masks
# ------------------------------------------------------------

train_mask = torch.zeros(
    len(experiment_df),
    dtype=torch.bool
)

test_mask = torch.zeros(
    len(experiment_df),
    dtype=torch.bool
)

train_mask[train_node_indices] = True
test_mask[test_node_indices] = True

# ------------------------------------------------------------
# Update graph
# ------------------------------------------------------------

graph_data.train_mask = train_mask.to(device)
graph_data.test_mask = test_mask.to(device)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

train_labels = (
    graph_data.y[graph_data.train_mask]
    .cpu()
    .numpy()
)

test_labels = (
    graph_data.y[graph_data.test_mask]
    .cpu()
    .numpy()
)

print("=" * 55)
print("CORRECTED GRAPH MASKS")
print("=" * 55)

print("Total nodes:", len(experiment_df))

print("\nTrain nodes:", len(train_labels))
print("Train distribution:")
print(pd.Series(train_labels).value_counts().sort_index())

print("\nTest nodes:", len(test_labels))
print("Test distribution:")
print(pd.Series(test_labels).value_counts().sort_index())

print("\nExpected:")
print("Train: 4000 Fake + 4000 Real")
print("Test : 1000 Fake + 1000 Real")

CORRECTED GRAPH MASKS
Total nodes: 10000

Train nodes: 8000
Train distribution:
0    4000
1    4000
Name: count, dtype: int64

Test nodes: 2000
Test distribution:
0    1000
1    1000
Name: count, dtype: int64

Expected:
Train: 4000 Fake + 4000 Real
Test : 1000 Fake + 1000 Real


In [ ]:
# ============================================================
# CELL 15: RETRAIN GRAPHSAGE WITH CORRECT MASKS
# ============================================================

import time

EPOCHS = 50

train_losses = []
train_accuracies = []

# Reinitialize model from scratch
gnn_model = GraphSAGE(
    input_dim=768,
    hidden_dim=128,
    output_dim=2,
    dropout=0.3
).to(device)

# Optimizer
optimizer = torch.optim.Adam(
    gnn_model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

criterion = torch.nn.CrossEntropyLoss()

print("=" * 55)
print("GRAPHSAGE TRAINING - CORRECTED")
print("=" * 55)

start_time = time.time()

for epoch in range(1, EPOCHS + 1):

    gnn_model.train()

    optimizer.zero_grad()

    output = gnn_model(
        graph_data.x,
        graph_data.edge_index
    )

    loss = criterion(
        output[graph_data.train_mask],
        graph_data.y[graph_data.train_mask]
    )

    loss.backward()

    optimizer.step()

    # Training accuracy
    predictions = output.argmax(dim=1)

    train_correct = (
        predictions[graph_data.train_mask]
        == graph_data.y[graph_data.train_mask]
    ).sum().item()

    train_total = int(
        graph_data.train_mask.sum()
    )

    train_accuracy = train_correct / train_total

    train_losses.append(loss.item())
    train_accuracies.append(train_accuracy)

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Loss: {loss.item():.4f} | "
            f"Train Accuracy: {train_accuracy:.4f}"
        )

training_time = time.time() - start_time

print("\n" + "=" * 55)
print("CORRECTED TRAINING COMPLETE")
print("=" * 55)

print(f"Training time: {training_time:.2f} seconds")
print(f"Final training loss: {train_losses[-1]:.4f}")
print(
    f"Final training accuracy: "
    f"{train_accuracies[-1] * 100:.2f}%"
)

GRAPHSAGE TRAINING - CORRECTED
Epoch 01/50 | Loss: 0.6951 | Train Accuracy: 0.5000
Epoch 05/50 | Loss: 0.5852 | Train Accuracy: 0.8063
Epoch 10/50 | Loss: 0.4380 | Train Accuracy: 0.8181
Epoch 15/50 | Loss: 0.3905 | Train Accuracy: 0.8315
Epoch 20/50 | Loss: 0.3498 | Train Accuracy: 0.8496
Epoch 25/50 | Loss: 0.3145 | Train Accuracy: 0.8712
Epoch 30/50 | Loss: 0.2957 | Train Accuracy: 0.8784
Epoch 35/50 | Loss: 0.2813 | Train Accuracy: 0.8809
Epoch 40/50 | Loss: 0.2650 | Train Accuracy: 0.8914
Epoch 45/50 | Loss: 0.2524 | Train Accuracy: 0.8985
Epoch 50/50 | Loss: 0.2423 | Train Accuracy: 0.9028

CORRECTED TRAINING COMPLETE
Training time: 1.00 seconds
Final training loss: 0.2423
Final training accuracy: 90.28%


In [ ]:
# ============================================================
# CELL 16: FINAL GRAPHSAGE TEST EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Evaluation mode
gnn_model.eval()

with torch.no_grad():

    test_output = gnn_model(
        graph_data.x,
        graph_data.edge_index
    )

    test_predictions = test_output.argmax(dim=1)

# ------------------------------------------------------------
# Get ONLY the correctly defined test nodes
# ------------------------------------------------------------

y_test = (
    graph_data.y[graph_data.test_mask]
    .cpu()
    .numpy()
)

y_pred = (
    test_predictions[graph_data.test_mask]
    .cpu()
    .numpy()
)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

test_accuracy = accuracy_score(y_test, y_pred)

test_precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("=" * 55)
print("FINAL GRAPHSAGE TEST RESULTS")
print("=" * 55)

print(f"Test samples: {len(y_test)}")
print(f"Accuracy : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Precision: {test_precision:.4f} ({test_precision*100:.2f}%)")
print(f"Recall   : {test_recall:.4f} ({test_recall*100:.2f}%)")
print(f"F1 Score : {test_f1:.4f} ({test_f1*100:.2f}%)")

print("\n" + "=" * 55)
print("CLASSIFICATION REPORT")
print("=" * 55)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Fake", "Real"],
        zero_division=0
    )
)

FINAL GRAPHSAGE TEST RESULTS
Test samples: 2000
Accuracy : 0.9025 (90.25%)
Precision: 0.8942 (89.42%)
Recall   : 0.9130 (91.30%)
F1 Score : 0.9035 (90.35%)

CLASSIFICATION REPORT
              precision    recall  f1-score   support

        Fake       0.91      0.89      0.90      1000
        Real       0.89      0.91      0.90      1000

    accuracy                           0.90      2000
   macro avg       0.90      0.90      0.90      2000
weighted avg       0.90      0.90      0.90      2000



In [ ]:
# ============================================================
# CELL 17: SAVE FINAL GNN MODEL AND RESULTS
# ============================================================

import json
import pandas as pd
import torch
import os

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

MODEL_PATH = f"{PROJECT_DIR}/models/llm_gnn_graphsage.pth"
RESULTS_PATH = f"{PROJECT_DIR}/results/llm_gnn_results.csv"
CONFIG_PATH = f"{PROJECT_DIR}/results/llm_gnn_config.json"

# ------------------------------------------------------------
# Save model
# ------------------------------------------------------------

torch.save(
    {
        "model_state_dict": gnn_model.state_dict(),
        "input_dim": 768,
        "hidden_dim": 128,
        "output_dim": 2,
        "dropout": 0.3
    },
    MODEL_PATH
)

# ------------------------------------------------------------
# Save metrics
# ------------------------------------------------------------

results_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Score": [
        test_accuracy,
        test_precision,
        test_recall,
        test_f1
    ]
})

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

# ------------------------------------------------------------
# Save experiment configuration
# ------------------------------------------------------------

config = {
    "dataset": "WELFake",
    "total_original_articles": 72134,
    "articles_after_cleaning": 72095,
    "experimental_articles": 10000,
    "train_articles": 8000,
    "test_articles": 2000,
    "embedding_model": "distilbert-base-uncased",
    "embedding_dimension": 768,
    "max_tokens": 256,
    "graph_method": "k-Nearest Neighbors",
    "k": 10,
    "similarity": "cosine",
    "gnn": "GraphSAGE",
    "hidden_dimension": 128,
    "second_layer_dimension": 64,
    "epochs": 50,
    "learning_rate": 0.001,
    "weight_decay": 0.0005,
    "accuracy": float(test_accuracy),
    "precision": float(test_precision),
    "recall": float(test_recall),
    "f1_score": float(test_f1)
}

with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=4)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("=" * 55)
print("FINAL LLM + GNN CHECKPOINT")
print("=" * 55)

print("Model saved:", os.path.exists(MODEL_PATH))
print("Results saved:", os.path.exists(RESULTS_PATH))
print("Configuration saved:", os.path.exists(CONFIG_PATH))

print("\nModel:", MODEL_PATH)
print("Results:", RESULTS_PATH)
print("Config:", CONFIG_PATH)

print("\nFINAL TEST PERFORMANCE")
print(f"Accuracy : {test_accuracy*100:.2f}%")
print(f"Precision: {test_precision*100:.2f}%")
print(f"Recall   : {test_recall*100:.2f}%")
print(f"F1 Score : {test_f1*100:.2f}%")

FINAL LLM + GNN CHECKPOINT
Model saved: True
Results saved: True
Configuration saved: True

Model: /content/drive/MyDrive/FakeNewsProject/models/llm_gnn_graphsage.pth
Results: /content/drive/MyDrive/FakeNewsProject/results/llm_gnn_results.csv
Config: /content/drive/MyDrive/FakeNewsProject/results/llm_gnn_config.json

FINAL TEST PERFORMANCE
Accuracy : 90.25%
Precision: 89.42%
Recall   : 91.30%
F1 Score : 90.35%
